# 03 - Security Testing & Adversarial Robustness
**Stage 6 of the SOC-Copilot project.**

Stage 5 measured *average* performance on a random test split.
Stage 6 asks the harder question: **how does the system behave under attack?**

We probe four threat scenarios that map to real adversary playbooks:

| # | Scenario | MITRE | What the attacker is trying to do |
|---|---|---|---|
| A | Burst brute force | T1110.001 | Smash through `admin` with 50 failed attempts in 5 minutes |
| B | Low-and-slow spray | T1110.003 | Hit 30 different users from one IP with only 2 fails each |
| C | Valid-account abuse | T1078 | Use stolen creds to log in successfully from an unknown IP at 02:14 |
| D | **Evasion** - attacker mimics business hours and known-IP shape | adversarial T1078 | Spoof `hour` and bytes/duration to look like a normal user |

For each scenario we check:
1. Does the **Isolation Forest detector** flag the events?
2. Does the **MITRE mapper** assign the correct technique ID?
3. Where does the system fail, and what would we change in production?

We also include a small **adversarial perturbation** experiment to show how the model degrades when the attacker has feature-level knowledge.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "Project"
sys.path.insert(0, str(PROJECT_ROOT.parent))

from Project.src.mitre_mapper import map_event

artefact = joblib.load(PROJECT_ROOT / "models" / "isolation_forest.joblib")
model         = artefact["model"]
scaler        = artefact["scaler"]
feature_cols  = artefact["feature_columns"]
print("Loaded model with", len(feature_cols), "features")

Loaded model with 11 features


## 1. Helpers

In [2]:
SERVICES = ["ssh", "rdp", "vpn", "webapp"]

def to_feature_row(ev: dict) -> np.ndarray:
    row = {c: 0 for c in feature_cols}
    for k in ["hour", "failed_attempts_5min", "success",
              "bytes_sent", "session_duration_ms",
              "user_is_known", "ip_is_known"]:
        row[k] = ev[k]
    svc_col = f"svc_{ev['service']}"
    if svc_col in row: row[svc_col] = 1
    return np.array([row[c] for c in feature_cols], dtype=float)

def score_events(events):
    X = np.vstack([to_feature_row(e) for e in events])
    Xs = scaler.transform(X)
    preds  = (model.predict(Xs) == -1).astype(int)
    scores = -model.score_samples(Xs)
    return preds, scores

## 2. Scenario A - burst brute force (T1110.001)

50 failed `ssh` logins against `admin` from an external IP at 03:00.
This is the easiest case: the model should catch ~all of them.

In [3]:
def burst_brute_force(n=50, seed=1):
    rng = np.random.default_rng(seed)
    events = []
    for i in range(n):
        events.append({
            "hour": 3,
            "failed_attempts_5min": int(rng.integers(15, 60)),
            "success": 0,
            "bytes_sent": int(rng.integers(80, 400)),
            "session_duration_ms": int(rng.integers(200, 2000)),
            "user_is_known": 1,
            "ip_is_known": 0,
            "service": "ssh",
            "user": "admin",
            "source_ip": "185.243.115.84",
        })
    return events

evA = burst_brute_force()
predA, scoreA = score_events(evA)
print(f"Detection rate: {predA.mean():.2%}  ({predA.sum()}/{len(predA)})")
print(f"Mean anomaly score: {scoreA.mean():.3f}")

# Mapper sanity check on first event
example_alert = map_event(evA[0], anomaly_score=float(scoreA[0]))
print(f"\nMapper output for event #0:")
print(f"  technique : {example_alert.technique_id} -- {example_alert.technique_name}")
print(f"  confidence: {example_alert.confidence}")
print(f"  rationale : {example_alert.rationale}")

Detection rate: 100.00%  (50/50)
Mean anomaly score: 0.747

Mapper output for event #0:
  technique : T1110.001 -- Brute Force: Password Guessing
  confidence: high
  rationale : ['36 failed logins in a 5-minute window (threshold = 10)', 'source IP 185.243.115.84 is not in the known-good list', 'attempt did not succeed (consistent with guessing)']


## 3. Scenario B - low-and-slow password spray (T1110.003)

30 different users, one external IP, 2 failed attempts each, in business hours.
The per-event signal is weak; the model has to lean on `ip_is_known=0`.

In [4]:
def password_spray(n_users=30, seed=2):
    rng = np.random.default_rng(seed)
    events = []
    for _ in range(n_users):
        events.append({
            "hour": int(rng.integers(9, 17)),
            "failed_attempts_5min": int(rng.integers(1, 4)),
            "success": 0,
            "bytes_sent": int(rng.integers(300, 900)),
            "session_duration_ms": int(rng.integers(500, 3000)),
            "user_is_known": 1,
            "ip_is_known": 0,
            "service": rng.choice(["vpn", "webapp"]),
            "user": f"user_{_}",
            "source_ip": "91.219.236.222",
        })
    return events

evB = password_spray()
predB, scoreB = score_events(evB)
print(f"Detection rate: {predB.mean():.2%}  ({predB.sum()}/{len(predB)})")

mapper_techs = [map_event(e, float(s)).technique_id for e, s in zip(evB, scoreB)]
print(f"Mapper distribution: {pd.Series(mapper_techs).value_counts().to_dict()}")

Detection rate: 100.00%  (30/30)
Mapper distribution: {'T1110.003': 30}


## 4. Scenario C - valid-account abuse (T1078)

Successful login, known user, unknown IP, 02:14.
This is the hardest unsupervised case. The only signal is `ip_is_known=0`.

In [5]:
def valid_account_abuse(n=20, seed=3):
    rng = np.random.default_rng(seed)
    events = []
    for _ in range(n):
        events.append({
            "hour": int(rng.choice([0,1,2,3,4,23])),
            "failed_attempts_5min": 0,
            "success": 1,
            "bytes_sent": int(rng.integers(5000, 12000)),
            "session_duration_ms": int(rng.integers(300_000, 900_000)),
            "user_is_known": 1,
            "ip_is_known": 0,
            "service": rng.choice(["vpn", "rdp"]),
            "user": "alice",
            "source_ip": "45.83.64.219",
        })
    return events

evC = valid_account_abuse()
predC, scoreC = score_events(evC)
print(f"Detection rate: {predC.mean():.2%}  ({predC.sum()}/{len(predC)})")
mapper_techs = [map_event(e, float(s)).technique_id for e, s in zip(evC, scoreC)]
print(f"Mapper distribution: {pd.Series(mapper_techs).value_counts().to_dict()}")

Detection rate: 65.00%  (13/20)
Mapper distribution: {'T1078': 20}


## 5. Scenario D - evasion (adversarial T1078)

The attacker has read our blog post and now knows what features we look at.
They mimic:
  - business-hour login (`hour=14`)
  - normal byte volume
  - normal session length
  - failed-attempt count = 0

But they cannot fake `ip_is_known=1` without first compromising a known device.
This scenario stresses the *single feature that the attacker cannot easily forge*.


In [6]:
def evasive_t1078(n=20, seed=4):
    rng = np.random.default_rng(seed)
    events = []
    for _ in range(n):
        events.append({
            "hour": int(rng.choice([10, 11, 13, 14, 15])),     # office hours
            "failed_attempts_5min": 0,
            "success": 1,
            "bytes_sent": int(rng.normal(4000, 1000)),          # normal-looking
            "session_duration_ms": int(rng.normal(180_000, 60_000)),
            "user_is_known": 1,
            "ip_is_known": 0,                                   # cannot fake
            "service": "vpn",
            "user": "bob",
            "source_ip": "5.188.206.13",
        })
    return events

evD = evasive_t1078()
predD, scoreD = score_events(evD)
print(f"Detection rate: {predD.mean():.2%}  ({predD.sum()}/{len(predD)})")
mapper_techs = [map_event(e, float(s)).technique_id for e, s in zip(evD, scoreD)]
print(f"Mapper distribution: {pd.Series(mapper_techs).value_counts().to_dict()}")

Detection rate: 0.00%  (0/20)
Mapper distribution: {'T1078': 20}


## 6. Adversarial perturbation -- what if `ip_is_known` is wrong?

In production, `ip_is_known` is computed from a reputation list. If the
attacker compromises a previously-trusted residential IP (e.g. via SOHO router
abuse -- exactly the Volt Typhoon tradecraft from Lab 1), the list lies and
the flag becomes 1.

Let's measure the impact: what happens to every scenario above if we flip
`ip_is_known` from 0 to 1?

In [7]:
def with_known_ip(events):
    return [dict(e, ip_is_known=1) for e in events]

rows = []
for name, evs in [("A burst BF", evA), ("B spray", evB), ("C T1078", evC), ("D evasive", evD)]:
    p_real, _ = score_events(evs)
    p_flip, _ = score_events(with_known_ip(evs))
    rows.append({
        "scenario": name,
        "detect_rate (real ip_known=0)": f"{p_real.mean():.0%}",
        "detect_rate (attacker spoofs ip_known=1)": f"{p_flip.mean():.0%}",
        "degradation": f"-{(p_real.mean() - p_flip.mean()) * 100:.0f} pts",
    })
pd.DataFrame(rows)

,scenario,detect_rate (real ip_known=0),detect_rate (attacker spoofs ip_known=1),degradation
0,A burst BF,100%,100%,-0 pts
1,B spray,100%,100%,-0 pts
2,C T1078,65%,65%,-0 pts
3,D evasive,0%,0%,-0 pts


**Reading the table:** the model collapses on T1078 and the evasive scenario the moment the attacker can land on a previously-trusted IP. That is the *known limitation* the system must surface in `docs/limitations.md`, and it is the reason the LLM Triage Agent labels the confidence as `low` rather than `high` when the rule chain is thin.

## 7. Summary

- The detector handles **loud** brute force essentially perfectly (the easy case).
- The detector handles **low-and-slow spraying** because the spraying source IPs are still unknown.
- **Valid-account abuse** is the documented weak spot - it relies on a single feature.
- A determined attacker who first compromises a trusted IP (SOHO-router pivot, à la Volt Typhoon) defeats the per-event detector. The mitigation is *not* a better single-event model; it is a higher-order detector that looks at *cross-user behaviour* (impossible-travel, first-login-from-this-IP-ever, etc.). That is future work.

These trade-offs are exactly what the LLM Triage Agent communicates to the analyst - see notebook outputs feeding into `app/app.py`.